# Task 1.2 — Key Assumptions (8 marks)

**Paper**: *Learning to Grade Short Answer Questions using Semantic Similarity Measures and Dependency Graph Alignments* (Mohler, Bunescu & Mihalcea, ACL 2011)

---

## Assumption 1

**Assumption**: Lexical and semantic overlap between a student answer and the reference answer is a reliable indicator of answer quality (i.e., higher textual similarity → higher grade).

**Why the method needs it**: The entire feature engineering pipeline — TF-IDF cosine similarity, WordNet-based measures, LSA/ESA embeddings, and word overlap — assumes that a correct student answer will share substantial vocabulary and meaning with the reference. The SVR model learns grade predictions as a weighted combination of these similarity features (Section 3.4, $g(A_i, A_s) = u^T \psi(A_i, A_s)$), so if similarity does not correlate with correctness, the learned weights become meaningless.

**Violation scenario**: A student provides a perfectly correct answer using entirely different vocabulary or domain-specific jargon not present in the reference answer (heavy paraphrasing). For example, if the reference says "to simulate the behavior of portions of the desired software product" and the student says "to trial a scaled-down mock-up of the planned application," the lexical overlap is near zero despite equivalent meaning. The model would assign an inappropriately low grade.

**Paper reference**: Section 3.3 (all 11 BOW similarity measures assume overlap = quality); Table 5 shows correlation between individual similarity measures and grades, implicitly assuming this relationship holds.

---

## Assumption 2

**Assumption**: Dependency parse structure provides additional grading-relevant information beyond what bag-of-words features capture — specifically, that syntactic role alignment (subject-verb-object relationships) helps distinguish correct from incorrect answers.

**Why the method needs it**: The node-to-node matching stage (Section 3.1) and graph alignment via the Hungarian algorithm (Section 3.2) are computationally expensive components that are only justified if syntactic structure adds discriminative power beyond BOW features. The paper's 68-feature perceptron includes role-based features (RoleBased, VerbsSubject, VerbsObject in Table 2) that are meaningful only if syntactic roles matter for grading.

**Violation scenario**: For questions whose correct answers are primarily a list of keywords or concepts (e.g., "Name the advantages of OOP: abstraction, reusability, modularity"), syntactic structure is essentially irrelevant — any permutation or rephrasing of the list is equally correct. In such cases, the dependency alignment adds noise rather than signal, and simpler BOW approaches would suffice.

**Paper reference**: Section 3.1 ("we employ a rudimentary dependency-graph alignment module"); Table 7 shows the Hybrid model (BOW + Alignment) improves over BOW-only, but only marginally for SVR (ρ = 0.464 vs. 0.431), implying the assumption holds weakly.

---

## Assumption 3

**Assumption**: A single instructor-provided reference answer is sufficient to represent the space of all correct responses for a given question.

**Why the method needs it**: The grading pipeline compares each student answer against exactly one reference answer $A_i$ (Section 3, Figure 1). All similarity computations — node matching, graph alignment, and BOW features — produce scores relative to this single reference. If the space of correct answers is much broader than what one reference captures, the model systematically penalises valid but differently-structured responses.

**Violation scenario**: Open-ended or design-type questions where multiple fundamentally different approaches are equally valid. For instance, asking "How would you implement a search algorithm?" could have correct answers ranging from binary search to hash-table lookup to BFS/DFS, each with entirely different vocabulary and structure. A single reference answer would fail to cover this diversity.

**Paper reference**: Section 4 describes the dataset as having one instructor answer per question; Section 2 notes that Mohler & Mihalcea (2009) used a "relevance feedback" approach to augment the reference, acknowledging this limitation implicitly.